# Cell types by distance from the detected plaques

Reads the h5ad written by `08_plaque_detected_alpha_pipeline.ipynb` and plots the
cell-type composition of each plaque-distance range.

Distances are signed distances to the **alpha-shape plaque circumference**: negative
inside a plaque, 0 on the boundary, positive outside. The cut-offs are read from
`uns['plaque_proximity']`, so the figures follow whatever was analysed.

## 1. Load the saved object

In [ ]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from anndata.io import read_elem

H5AD_PATH = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/"
    "AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/"
    "RESULTS/20240627__192310__KAECH_AD_GBM_240627/08_Plaque_Proximity_Analysis/"
    "adata_combined_with_alphashape_plaque_edge_15_20_30_40um.h5ad"
)

FIGURE_DIR = H5AD_PATH.parent / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# These barplots only need obs and the proximity metadata, so read just those
# elements rather than the whole 120 MB object. Swap in
#     adata = ad.read_h5ad(H5AD_PATH)
#     cell_metadata = adata.obs
#     proximity_information = adata.uns["plaque_proximity"]
# if you want the full object in memory for other work in this notebook.
with h5py.File(H5AD_PATH, "r") as handle:
    cell_metadata = read_elem(handle["obs"])
    proximity_information = read_elem(handle["uns"]["plaque_proximity"])

print(cell_metadata.shape[0], "cells,", cell_metadata.shape[1], "obs columns")
print("Method:", proximity_information["method"])
print("Cut-offs:", proximity_information["cutoff_edges_um"], "µm")
print("Plaques:", int(proximity_information["n_plaques"]))

## 2. Build the counts and composition tables

`USE_EXCLUSIVE_RANGES` switches between the two readings of "range":

- `False` (default): **cumulative** - `<15`, `<20`, `<30`, `<40`. Nested, so every cell
  in `<15` is also in `<20`. Matches the `within_*um_of_plaque` columns.
- `True`: **exclusive** - `<15`, `15-20`, `20-30`, `30-40`, `>=40`. Non-overlapping, so
  the percentages describe a true distance profile.

In [ ]:
CELLTYPE_KEY = "manual_celltype"

# Cell types to leave out of the figures entirely.
DROP_CELLTYPES = ["unknown (REMOVE)"]

# False: cumulative ranges (<15, <20, <30, <40) - nested, each a superset.
# True:  exclusive ranges (<15, 15-20, 20-30, 30-40, >=40) - non-overlapping.
USE_EXCLUSIVE_RANGES = False

# Cut-offs come from the file, so the figures follow whatever was analysed.
BAND_EDGES_UM = [float(edge) for edge in proximity_information["cutoff_edges_um"]]

# Only cells that were actually scored against the IF plaque image: this drops
# the mock section and any infected cell whose coordinates fell off the image.
scored = cell_metadata["plaque_proximity_valid"].to_numpy().astype(bool)
cells = cell_metadata.loc[
    scored, [CELLTYPE_KEY, "plaque_edge_distance_um", "inside_plaque"]
].copy()
cells[CELLTYPE_KEY] = cells[CELLTYPE_KEY].astype(str)
cells = cells.loc[~cells[CELLTYPE_KEY].isin(DROP_CELLTYPES)]

distance_um = cells["plaque_edge_distance_um"].to_numpy()

# One boolean mask per range, plus an all-cells reference to compare against.
range_masks = {}
if USE_EXCLUSIVE_RANGES:
    for lower, upper in zip([-np.inf, *BAND_EDGES_UM], BAND_EDGES_UM):
        label = f"<{upper:g}" if np.isinf(lower) else f"{lower:g}-{upper:g}"
        range_masks[label] = (distance_um >= lower) & (distance_um < upper)
    range_masks[f">={BAND_EDGES_UM[-1]:g}"] = distance_um >= BAND_EDGES_UM[-1]
else:
    for upper in BAND_EDGES_UM:
        range_masks[f"<{upper:g}"] = distance_um < upper

REFERENCE_LABEL = "all scored cells"
range_masks[REFERENCE_LABEL] = np.ones(len(cells), dtype=bool)
range_labels = list(range_masks)

# Cell types ordered by overall abundance, so the bars read top-down.
celltype_order = cells[CELLTYPE_KEY].value_counts().index.tolist()

counts = pd.DataFrame(
    {
        label: cells.loc[mask, CELLTYPE_KEY].value_counts().reindex(celltype_order)
        for label, mask in range_masks.items()
    },
    index=celltype_order,
).fillna(0).astype(int)

# Composition: what share of the cells in this range is each cell type.
percent = 100 * counts.div(counts.sum(axis=0), axis=1)

print("Cells per range:")
print(counts.sum(axis=0).to_string())
print()
print("Cell counts by type and range")
display(counts)
print("Composition (% of the cells in each range)")
display(percent.round(2))

## 3. Composition of each range

In [ ]:
# Ordinal blue ramp: the ranges are ordered by distance, so one hue stepped
# light -> dark, not a categorical palette. The reference bar is neutral gray
# because it is a baseline, not another distance.
RANGE_COLORS = {
    label: color
    for label, color in zip(
        [label for label in range_labels if label != REFERENCE_LABEL],
        ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"],
    )
}
RANGE_COLORS[REFERENCE_LABEL] = "#9a9992"

TEXT_PRIMARY = "#0b0b0b"
TEXT_SECONDARY = "#52514e"
GRID_COLOR = "#e3e2dd"


def style_axis(ax, xlabel):
    """Recessive grid and axes; text wears ink, never the series colour."""
    ax.set_xlabel(xlabel, fontsize=10, color=TEXT_SECONDARY)
    ax.xaxis.grid(True, color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.yaxis.grid(False)
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.tick_params(axis="both", colors=TEXT_SECONDARY, length=0, labelsize=9)


range_totals = counts.sum(axis=0)
bar_positions = np.arange(len(celltype_order))
group_height = 0.82 / len(range_labels)

fig, ax = plt.subplots(figsize=(11, 0.5 * len(celltype_order) + 2))

for position, label in enumerate(range_labels):
    offset = (position - (len(range_labels) - 1) / 2) * group_height
    # Range totals go in the legend, so no second panel is needed for them.
    suffix = f"  (n={range_totals[label]:,})"
    legend_label = (
        label + suffix if label == REFERENCE_LABEL else f"{label} µm" + suffix
    )
    ax.barh(
        bar_positions + offset,
        percent[label].to_numpy(),
        height=group_height * 0.86,   # leaves a surface gap between fills
        color=RANGE_COLORS[label],
        label=legend_label,
        zorder=3,
    )

ax.set_yticks(bar_positions)
ax.set_yticklabels(celltype_order, fontsize=9, color=TEXT_PRIMARY)
ax.invert_yaxis()
style_axis(ax, "% of the cells in that range")

range_kind = "exclusive" if USE_EXCLUSIVE_RANGES else "cumulative"
ax.set_title(
    "Cell-type composition by distance from the plaque circumference\n"
    f"{range_kind} ranges, {int(proximity_information['n_plaques'])} alpha-shape plaques",
    fontsize=12, color=TEXT_PRIMARY, loc="left", pad=12,
)
ax.legend(
    frameon=False, fontsize=9, labelcolor=TEXT_SECONDARY,
    loc="lower right", title="distance range", title_fontsize=9,
)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "celltype_composition_by_plaque_distance.png",
    dpi=300, bbox_inches="tight", facecolor="white",
)
plt.show()

## 4. Enrichment closest to the plaques

Composition alone conflates "this cell type is common" with "this cell type is near
plaques". The log2 ratio against each type's share of all scored cells separates them.

In [ ]:
# Enrichment: is a cell type over- or under-represented close to plaques,
# relative to its share of all scored cells? That is a polarity question, so a
# diverging pair (blue / red) around a neutral zero.
ENRICHMENT_RANGE = list(range_masks)[0]

baseline_fraction = counts[REFERENCE_LABEL] / counts[REFERENCE_LABEL].sum()
range_fraction = counts[ENRICHMENT_RANGE] / counts[ENRICHMENT_RANGE].sum()

enrichment = pd.DataFrame(
    {
        "cells_in_range": counts[ENRICHMENT_RANGE],
        "percent_in_range": 100 * range_fraction,
        "percent_all_cells": 100 * baseline_fraction,
    }
)
enrichment["log2_enrichment"] = np.log2(
    range_fraction.replace(0, np.nan) / baseline_fraction
)

# Wilson 95% interval on the within-range share, mapped onto the ratio. The
# baseline is treated as fixed because it comes from far more cells.
range_total = int(counts[ENRICHMENT_RANGE].sum())
z = 1.959964
observed = range_fraction.to_numpy()
denominator = 1 + z**2 / range_total
center = (observed + z**2 / (2 * range_total)) / denominator
half_width = (
    z * np.sqrt(observed * (1 - observed) / range_total + z**2 / (4 * range_total**2))
) / denominator
enrichment["log2_low"] = np.log2(
    np.clip(center - half_width, 1e-9, None) / baseline_fraction
)
enrichment["log2_high"] = np.log2(
    np.clip(center + half_width, 1e-9, None) / baseline_fraction
)

enrichment = enrichment.sort_values("log2_enrichment")
interval_excludes_zero = (enrichment["log2_low"] > 0) | (enrichment["log2_high"] < 0)

POSITIVE_COLOR = "#2a78d6"
NEGATIVE_COLOR = "#e34948"
NEUTRAL_COLOR = "#c6c5bf"

bar_colors = [
    (POSITIVE_COLOR if value > 0 else NEGATIVE_COLOR) if excludes_zero else NEUTRAL_COLOR
    for value, excludes_zero in zip(
        enrichment["log2_enrichment"], interval_excludes_zero
    )
]

fig, ax = plt.subplots(figsize=(9, 0.42 * len(enrichment) + 2))
positions = np.arange(len(enrichment))

ax.barh(positions, enrichment["log2_enrichment"], height=0.66, color=bar_colors, zorder=3)
ax.errorbar(
    enrichment["log2_enrichment"],
    positions,
    xerr=[
        enrichment["log2_enrichment"] - enrichment["log2_low"],
        enrichment["log2_high"] - enrichment["log2_enrichment"],
    ],
    fmt="none", ecolor=TEXT_SECONDARY, elinewidth=1, capsize=2.5, zorder=4,
)
ax.axvline(0, color=TEXT_SECONDARY, linewidth=1, zorder=2)

ax.set_yticks(positions)
ax.set_yticklabels(enrichment.index, fontsize=9, color=TEXT_PRIMARY)
style_axis(ax, f"log2 enrichment within {ENRICHMENT_RANGE} µm vs all scored cells")
ax.set_title(
    f"Which cell types sit within {ENRICHMENT_RANGE} µm of a plaque",
    fontsize=12, color=TEXT_PRIMARY, loc="left", pad=12,
)
ax.text(
    0.01, 0.985,
    "blue enriched, red depleted, gray 95% CI crosses zero\n"
    "whiskers: Wilson 95% CI on the within-range share",
    transform=ax.transAxes, ha="left", va="top", fontsize=8, color=TEXT_SECONDARY,
)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "celltype_enrichment_near_plaques.png",
    dpi=300, bbox_inches="tight", facecolor="white",
)
plt.show()

display(enrichment.round(3))

## 5. Composition shift across the ranges

In [ ]:
# One line per cell type reads the distance trend more directly than 16 groups
# of bars. Show the cell types that actually shift, not just the abundant ones,
# and only those with enough cells in the widest range to be stable.
MINIMUM_CELLS_FOR_TREND = 50
N_TREND_CELLTYPES = 6

plotted_range_labels = [label for label in range_labels if label != REFERENCE_LABEL]

shift_vs_baseline = (
    percent[plotted_range_labels[0]] / percent[REFERENCE_LABEL].replace(0, np.nan)
).apply(np.log2).abs()
eligible = counts[plotted_range_labels[-1]] >= MINIMUM_CELLS_FOR_TREND
trend_celltypes = (
    shift_vs_baseline.loc[eligible]
    .sort_values(ascending=False)
    .head(N_TREND_CELLTYPES)
    .index.tolist()
)

# Categorical hues in the documented fixed order, plus a marker per series so
# identity never rests on colour alone where lines cross.
LINE_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
LINE_MARKERS = ["o", "s", "^", "D", "v", "P"]

fig, ax = plt.subplots(figsize=(10, 6))
x_positions = np.arange(len(plotted_range_labels))

for celltype, color, marker in zip(trend_celltypes, LINE_COLORS, LINE_MARKERS):
    values = percent.loc[celltype, plotted_range_labels].to_numpy()
    baseline = percent.loc[celltype, REFERENCE_LABEL]
    ax.axhline(baseline, color=color, linewidth=0.7, linestyle=":", alpha=0.35, zorder=1)
    ax.plot(
        x_positions, values, marker=marker, markersize=8, linewidth=2, color=color,
        markeredgecolor="white", markeredgewidth=1.2,
        label=f"{celltype}  ({baseline:.1f}% overall)", zorder=3,
    )

ax.set_xticks(x_positions)
ax.set_xticklabels([f"{label} µm" for label in plotted_range_labels], fontsize=9)
style_axis(
    ax,
    ("exclusive" if USE_EXCLUSIVE_RANGES else "cumulative")
    + " distance range from the plaque circumference",
)
ax.set_ylabel("% of the cells in that range", fontsize=10, color=TEXT_SECONDARY)
ax.set_title(
    "Composition shift with distance\n"
    "dotted line = that cell type's share of all scored cells",
    fontsize=12, color=TEXT_PRIMARY, loc="left", pad=12,
)
ax.legend(
    frameon=False, fontsize=9, labelcolor=TEXT_SECONDARY,
    bbox_to_anchor=(1.01, 1), loc="upper left",
)
ax.set_xlim(-0.2, len(plotted_range_labels) - 0.8)
ax.set_ylim(0, None)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "celltype_composition_trend.png",
    dpi=300, bbox_inches="tight", facecolor="white",
)
plt.show()